# Benchmark DeepGaze MSDB on all of UEyes, on a free Colab GPU

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

UEyes (Jiang et al., CHI 2023, CC BY 4.0) is downloaded straight from Zenodo into this session; only the images, gaze logs and split file are unpacked. Results are cached per image, so if a cell is interrupted, run it again and it continues.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.environ['MA_BENCH'] = '/content/bench'

In [ ]:
!git clone -q https://github.com/pranikz/MediaAttention /content/MediaAttention 2>/dev/null || git -C /content/MediaAttention pull -q
!pip install -q -e "/content/MediaAttention[unisal]" "git+https://github.com/matthias-k/pysaliency.git@dev" pandas scikit-image einops clip-anytorch fire "deepgaze_pytorch @ git+https://github.com/matthias-k/DeepGaze"

In [ ]:
%%bash
set -e
D=/content/bench/data/ueyes
mkdir -p $D && cd $D
if [ ! -f UEyes_dataset/image_types.csv ]; then
  wget -q --show-progress -O ueyes.zip 'https://zenodo.org/api/records/8010312/files/UEyes_dataset.zip/content'
  unzip -q ueyes.zip 'UEyes_dataset/images/*' 'UEyes_dataset/eyetracker_logs/*' 'UEyes_dataset/image_types.csv'
  rm ueyes.zip
fi
ls UEyes_dataset/images | wc -l

In [ ]:
%cd /content/MediaAttention
!python benchmarks/run.py ueyes --models deepgaze_msdb 2>&1 | tr '\r' '\n' | awk 'NR % 100 == 0 || /done|Error|Traceback/'

In [ ]:
import json
print(json.dumps(json.load(open('benchmarks/results/ueyes.json'))['deepgaze_msdb'], indent=1))
!cd benchmarks/results && zip -q /content/msdb_ueyes_results.zip ueyes.json ueyes_per_image_deepgaze_msdb.csv && ls -la /content/msdb_ueyes_results.zip

Download `msdb_ueyes_results.zip` from the Files panel on the left (or run the next cell) and put both files in `benchmarks/results/`.

In [ ]:
from google.colab import files
files.download('/content/msdb_ueyes_results.zip')